In [ ]:
# v1
# v2 : added "if it likes it then has a higher chnace of liking it" (i assume randomness too much)
# v3 : changed it to +-0.1 pet like dislike (slow but nice randomness was too much i think?)
# v4 : just a combined plot +-1 yes


In [207]:
import csv
import matplotlib.pyplot as plt
import pandas as pd
df = pd.read_csv('activity_finished_log.csv', header=None, 
                 names=['timestamp' , 'pet_name', 'activity_name', 'partner_name', 'liked', 'activity_relationship', 'partner_relationship'])

df2 = df

In [208]:
df['timestamp'] = pd.to_datetime(df2['timestamp'], unit='ms')
df = df2[df2['partner_name'].notna()].copy()
df["partner_name"] = df["partner_name"].str.split("@", n=1).str[1]
df['pair'] = df['pet_name'] + "→" + df['partner_name']

heat = df.pivot_table(
    index='pair',
    columns='timestamp',
    values='partner_relationship',
    # aggfunc='last'
)
heat = heat.sort_index(axis=1)
heat = heat.ffill(axis=1)
heat.fillna(0, inplace=True)

import plotly.express as px

fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="RdYlGn",
    labels={
        "x": "Time",
        "y": "Relationship",
        "color": "Value"
    }
)

group_size = 5 

for i in range(group_size, len(heat.index), group_size):
    fig.add_hline(
        y=i - 0.5,
        line_width=5,
        line_color="black"
    )

fig.add_scatter(
    x=df['timestamp'],
    y=df['pair'],
    mode='markers',
    marker=dict(
        symbol='diamond',
        size=3,
        color='black'
    ),
    hovertext=df['activity_name'],
    hoverinfo='text'
)
fig.update_xaxes(range=[heat.columns.min(), heat.columns.max()])
# fig.update_yaxes(range= [heat.index.min(), heat.index.max()])

# fig.show()

fig.update_layout(height=900)

fig.show()

In [205]:
df["activity_pair"] = df["pet_name"] + " : " + df["activity_name"]

heat = df.pivot_table(
    index="activity_pair",
    columns="timestamp",
    values="activity_relationship",
    aggfunc="last"
)

heat = heat.sort_index(axis=1).ffill(axis=1)
heat.fillna(0, inplace=True)

fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="RdYlGn",
)

fig.update_layout(height=900)
fig.show()